# JobTech API → dlt → Snowflake: code walkthrough

This notebook breaks the script into small pieces so you can see **what each part receives, does, and returns**.

### Full flow

`query` → `params` → `jobads_resource()` → `_get_ads()` → JobTech API → JSON dictionary → `"hits"` list → `yield` one ad at a time → dlt → Snowflake

> In a normal `.py` script, `if __name__ == "__main__":` is the starting point when the file is run directly.  
> In a notebook, we do not need that block—we run cells manually from top to bottom.


## 1. Import the tools

These imports give Python the tools used by the original script.

In [1]:
import dlt
import requests
import json

## 2. Start with the values that would normally be inside `__main__`

In the script, these are the values we choose before calling `run_pipeline()`.

In [2]:
query = "data engineer"
table_name = "data_field_job_ads"

print("Query:", query)
print("Destination table:", table_name)

Query: data engineer
Destination table: data_field_job_ads


## 3. Build the `params` dictionary

This is the **request wish list**.

- `"q"` = what we want to search for
- `"limit"` = how many ads we want the API to return

Notice that the value stored in `query` becomes the value of `"q"`.

In [3]:
params = {
    "q": query,
    "limit": 3,   # small number for learning
}

print(params)
print("Value of q:", params["q"])
print("Value of limit:", params["limit"])

{'q': 'data engineer', 'limit': 3}
Value of q: data engineer
Value of limit: 3


## 4. Define the helper function `_get_ads()`

This is a **helper function** because its job is to handle the lower-level API work.

It receives:

1. `url_for_search`
2. `params`

Then it:

1. prepares the headers,
2. sends the GET request,
3. checks for HTTP errors,
4. converts the JSON response into Python objects,
5. returns a Python dictionary.

The leading `_` in `_get_ads` is a naming convention that means **internal/helper function**.

In [4]:
def _get_ads(url_for_search, params):
    headers = {"accept": "application/json"}

    response = requests.get(
        url_for_search,
        headers=headers,
        params=params
    )

    response.raise_for_status()

    return json.loads(response.content.decode("utf8"))

## 5. Call the helper ourselves

Before involving dlt, let's call `_get_ads()` directly so we can inspect what comes back.

In [5]:
url = "https://jobsearch.api.jobtechdev.se"
url_for_search = f"{url}/search"

json_response = _get_ads(url_for_search, params)

print("Python type:", type(json_response))
print("Top-level keys:", json_response.keys())

Python type: <class 'dict'>
Top-level keys: dict_keys(['total', 'positions', 'query_time_in_millis', 'result_time_in_millis', 'stats', 'freetext_concepts', 'hits'])


The API response is a **dictionary**.

A dictionary contains **key-value pairs**.

For example, the response may contain keys such as:

- `"total"`
- `"hits"`

`"hits"` is important because its value is a **list of job-ad dictionaries**.

In [6]:
hits = json_response["hits"]

print("Type of hits:", type(hits))
print("Number of ads returned:", len(hits))

Type of hits: <class 'list'>
Number of ads returned: 3


## 6. Look at one job ad

`hits` is a list.

Each item inside that list is one job ad, and each job ad is itself a dictionary.

In [7]:
first_ad = hits[0]

print("Type of one ad:", type(first_ad))
print("Some keys inside one ad:")
print(list(first_ad.keys())[:15])

Type of one ad: <class 'dict'>
Some keys inside one ad:
['relevance', 'id', 'external_id', 'original_id', 'label', 'webpage_url', 'logo_url', 'headline', 'application_deadline', 'number_of_vacancies', 'description', 'employment_type', 'salary_type', 'salary_description', 'duration']


Now we can access values using dictionary keys.

For example:

```python
first_ad["headline"]
```

means:

> Give me the value stored under the `"headline"` key.

And:

```python
first_ad["employer"]["name"]
```

means:

1. open the `"employer"` value,
2. that value is another dictionary,
3. then get its `"name"` value.


In [8]:
print("Headline:", first_ad["headline"])
print("Employer:", first_ad["employer"]["name"])

Headline: Data Engineer
Employer: AcadeMedia Support AB


## 7. Now add the dlt resource

This function connects the API helper to dlt.

Important flow:

```text
jobads_resource(params)
        ↓
_get_ads(url_for_search, params)
        ↓
API response dictionary
        ↓
["hits"]
        ↓
list of ads
        ↓
for ad in ...
        ↓
yield one ad at a time
```

`yield` is different from `return`.

- `return` gives back a result and ends the function.
- `yield` gives back one item, pauses, and continues later when the next item is requested.


In [9]:
@dlt.resource(write_disposition="replace")
def jobads_resource(params):
    url = "https://jobsearch.api.jobtechdev.se"
    url_for_search = f"{url}/search"

    for ad in _get_ads(url_for_search, params)["hits"]:
        yield ad

## 8. Test the resource without Snowflake

We can ask the resource for a few ads and inspect them before loading anything into Snowflake.

In [10]:
resource = jobads_resource(params=params)

for ad in resource:
    print(ad["headline"], "-", ad["employer"]["name"])

Data Engineer - AcadeMedia Support AB
Data Engineer - Luossavaara-Kiirunavaara Aktiebolag
Data Engineer - Techrytera AB


## 9. The pipeline function

The original `run_pipeline()` does three main things:

1. configures dlt,
2. builds the API `params`,
3. tells dlt to extract, normalize, and load the resource into Snowflake.

Notice the chain:

```python
pipeline.run(
    jobads_resource(params=params),
    table_name=table_name
)
```

The inner part:

```python
jobads_resource(params=params)
```

creates the dlt resource.

Then `pipeline.run(...)` starts consuming that resource.  
While dlt consumes it, the resource calls `_get_ads()`, receives the API response, enters the `"hits"` list, and yields the ads one by one.


In [ ]:
def run_pipeline(query, table_name):
    pipeline = dlt.pipeline(
        pipeline_name="jobsearch",
        destination="snowflake",
        dataset_name="staging",
    )

    params = {
        "q": query,
        "limit": 100,
    }

    load_info = pipeline.run(       # execution action
        jobads_resource(params=params),  #the data source with our applied filter which is the params
        table_name=table_name,    # where the data will go in snowflake inside the staging schema
    )

    print(load_info)  # the placeholder for what we have receieved

## 10. Optional: run the real Snowflake load

Only run the next cell when your dlt/Snowflake credentials are configured.

Your original script effectively does this when it reaches:

```python
run_pipeline(query, table_name)
```


In [ ]:
# Uncomment when you want to run the real pipeline:
# run_pipeline(query, table_name)

# Execution order of the original `.py` script

When you press **Run** on the original Python file, Python works through it like this:

1. **Imports** `dlt`, `requests`, `json`, `Path`, and `os`.
2. **Defines** `_get_ads()` — but does not run it yet.
3. **Defines** `jobads_resource()` — but does not fetch ads yet.
4. **Defines** `run_pipeline()` — but does not run it yet.
5. Reaches `if __name__ == "__main__":`.
6. Sets `query = "data engineer"`.
7. Sets `table_name = "data_field_job_ads"`.
8. Calls `run_pipeline(query, table_name)`.
9. `run_pipeline()` creates the dlt pipeline.
10. It creates `params = {"q": "data engineer", "limit": 100}`.
11. It gives `jobads_resource(params)` to `pipeline.run()`.
12. dlt starts consuming the resource.
13. `jobads_resource()` calls `_get_ads(url_for_search, params)`.
14. `_get_ads()` sends the request to JobTech.
15. JobTech returns JSON.
16. `_get_ads()` converts that JSON to a Python dictionary and returns it.
17. `jobads_resource()` takes the `"hits"` list.
18. It `yield`s each job ad to dlt.
19. dlt normalizes the nested data.
20. dlt loads the resulting tables into Snowflake.

### Mental model

Think of the three functions as different levels:

- `_get_ads()` → **How do I talk to the API?**
- `jobads_resource()` → **Which API data should I give to dlt?**
- `run_pipeline()` → **Where should dlt send that data?**

So the relationship is:

`run_pipeline()` → `jobads_resource()` → `_get_ads()`

The higher-level function uses the lower-level functions underneath it.
